# Special Tokens: BOS, EOS, PAD, UNK

A tokenizer turns text into token IDs. Besides normal tokens (pieces of words), it has a few **special tokens** that are not text. They are markers with a job:

| Token | Full name | Job |
|---|---|---|
| **BOS** | Beginning of sequence | Marks the start of the input text |
| **EOS** | End of sequence | Marks where the text ends; when the model generates it, generation stops |
| **PAD** | Padding | Filler that makes short sentences the same length as long ones in a batch |
| **UNK** | Unknown | Used for text the tokenizer cannot represent |

We use the **TinyLlama** tokenizer. We only load the tokenizer, not the model, so this notebook runs fast, even without a GPU.

## Step 1: Load the tokenizer

In [ ]:
from transformers import AutoTokenizer
# AutoTokenizer downloads the correct tokenizer for whichever model name we give it

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# the TinyLlama base model; we only need its tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# download the tokenizer files (the vocabulary and the splitting rules)

print("Vocabulary size:", len(tokenizer))  # how many different tokens this tokenizer knows (about 32,000)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Vocabulary size: 32000


## Step 2: See the special tokens and their IDs

In [ ]:
print("BOS:", tokenizer.bos_token, "→ ID", tokenizer.bos_token_id)
# the start-of-text marker and its number in the vocabulary

print("EOS:", tokenizer.eos_token, "→ ID", tokenizer.eos_token_id)
# the end-of-text marker and its number

print("UNK:", tokenizer.unk_token, "→ ID", tokenizer.unk_token_id)
# the unknown-text marker and its number

print("PAD:", tokenizer.pad_token, "→ ID", tokenizer.pad_token_id)
# the padding marker; "None" means TinyLlama was not given one

BOS: <s> → ID 1
EOS: </s> → ID 2
UNK: <unk> → ID 0
PAD: None → ID None


**What to notice:** special tokens have small ID numbers (0, 1, 2) because they were added to the vocabulary first. If PAD shows `None`, TinyLlama has no padding token of its own. We will fix that in Step 5.

## Step 3: BOS, the start marker

In [ ]:
text = "NAV is the price of one unit."  # a simple sentence to tokenize

ids = tokenizer.encode(text)  # convert the sentence into token IDs; the tokenizer adds BOS automatically
print("Token IDs:", ids)  # the first ID should be 1, which is BOS
print("Tokens:   ", tokenizer.convert_ids_to_tokens(ids))  # the same IDs shown as tokens; the first one should be <s>

ids_plain = tokenizer.encode(text, add_special_tokens=False)  # the same sentence, but told NOT to add special tokens
print("\nWithout special tokens:", tokenizer.convert_ids_to_tokens(ids_plain))  # now <s> is missing from the start


'''
Where BOS is decided

We don't set it anywhere in our code. The tokenizer's own settings decide it.
When the tokenizer was downloaded, it came with a settings file (tokenizer_config.json) that contains two lines:

add_bos_token: true    → add <s> at the start
add_eos_token: false   → do NOT add </s> at the end

tokenizer.encode(text) has a hidden default, add_special_tokens=True, which means "apply those settings". So:

encode(text)                            → applies settings → <s> added, </s> not added
encode(text, add_special_tokens=False)  → ignores settings → nothing added
Why EOS is not added

</s> means "the text is finished". When you give the model a prompt, you want it to continue writing.
If the tokenizer put </s> at the end of every prompt, the model would read it as "this text is already complete", and it would not answer properly.
'''

Token IDs: [1, 405, 7520, 338, 278, 8666, 310, 697, 5190, 29889]
Tokens:    ['<s>', '▁N', 'AV', '▁is', '▁the', '▁price', '▁of', '▁one', '▁unit', '.']

Without special tokens: ['▁N', 'AV', '▁is', '▁the', '▁price', '▁of', '▁one', '▁unit', '.']


**What to notice:** the tokenizer puts `<s>` (ID 1) at the start by itself. It does **not** add `</s>` at the end. The EOS token has to be added separately, which we do next.

`▁` at the start of a token means there was a space before it.

## Step 4: EOS, the end marker

In [ ]:
ids_with_eos = ids + [tokenizer.eos_token_id]  # add the EOS ID (2) to the end of our list of IDs
print("Tokens:", tokenizer.convert_ids_to_tokens(ids_with_eos))  # now the list starts with <s> and ends with </s>

print("\nDecoded, special tokens kept:   ", tokenizer.decode(ids_with_eos))  # IDs back to text, showing <s> and </s>
print("Decoded, special tokens removed:", tokenizer.decode(ids_with_eos, skip_special_tokens=True))
 # IDs back to text, hiding them

Tokens: ['<s>', '▁N', 'AV', '▁is', '▁the', '▁price', '▁of', '▁one', '▁unit', '.', '</s>']

Decoded, special tokens kept:    <s> NAV is the price of one unit.</s>
Decoded, special tokens removed: NAV is the price of one unit.


**Where EOS appears in real use:** in the chat format, every message ends with `</s>`. During training, the model learns to write `</s>` when its answer is finished. When it writes `</s>`, generation stops.

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")  # the chat model's tokenizer, which has the chat template

messages = [  # a short conversation: one user message and one assistant reply
    {"role": "user", "content": "What is NAV?"},                      # the question
    {"role": "assistant", "content": "NAV is the price of one unit."}, # the answer
]

print(chat_tokenizer.apply_chat_template(messages, tokenize=False))
# show the conversation in TinyLlama's chat format; look for </s> after each message

<|user|>
What is NAV?</s>
<|assistant|>
NAV is the price of one unit.</s>



## Step 5: PAD, the filler

In [ ]:
# TinyLlama has no padding token of its own (Step 2 printed PAD: None)

if tokenizer.pad_token is None:  # check: does TinyLlama have a padding token?
    tokenizer.pad_token = tokenizer.unk_token  # no, so reuse <unk> as filler; the model ignores padding, so any token works

tokenizer.padding_side = "right"  # put the filler at the END of short sentences (easier to read in this demo)

sentences = ["What is NAV?", "What is NAV in mutual funds?"]  # two sentences of different lengths
batch = tokenizer(sentences, padding=True)  # tokenize both together;
# padding=True fills the shorter one to match the longer one

for i in range(len(sentences)):  # go through each sentence in the batch
    print(f"\nSentence {i + 1}: {sentences[i]}")  # show the original sentence
    print("Tokens:        ", tokenizer.convert_ids_to_tokens(batch["input_ids"][i]))  # its tokens, including any padding
    print("input_ids:     ", batch["input_ids"][i])  # its token IDs; padding appears as the PAD token's ID
    print("attention_mask:", batch["attention_mask"][i])  # 1 = real token (read it), 0 = padding (ignore it)


Sentence 1: What is NAV?
Tokens:         ['<s>', '▁What', '▁is', '▁N', 'AV', '?', '<unk>', '<unk>', '<unk>', '<unk>']
input_ids:      [1, 1724, 338, 405, 7520, 29973, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0]

Sentence 2: What is NAV in mutual funds?
Tokens:         ['<s>', '▁What', '▁is', '▁N', 'AV', '▁in', '▁mut', 'ual', '▁funds', '?']
input_ids:      [1, 1724, 338, 405, 7520, 297, 5478, 950, 29199, 29973]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


**What to notice:**
- Sentence 1 is shorter, so filler tokens are added to its end until both rows have the same length.
- In the `attention_mask`, the filler positions are `0`. The model ignores them.
- Sentence 2 is the longest, so it gets no padding, and its mask is all `1`s.

When generating text with `generate()`, padding goes on the **left** instead (`padding_side = "left"`), so every sentence's last real token is at the end of its row.

## Step 6: UNK, the unknown marker

In [ ]:
print("UNK:", tokenizer.unk_token, "→ ID", tokenizer.unk_token_id)  # the unknown token and its ID (0)

odd_text = "NAV ₹ 😊"  # text with a currency symbol and an emoji, which may not be in the vocabulary
odd_ids = tokenizer.encode(odd_text, add_special_tokens=False)  # convert it into token IDs, without BOS
print("Tokens: ", tokenizer.convert_ids_to_tokens(odd_ids))  # characters not in the vocabulary appear as byte tokens like <0xE2>
print("Decoded:", tokenizer.decode(odd_ids))  # decoding joins the bytes back together and gives the original text

UNK: <unk> → ID 0
Tokens:  ['▁N', 'AV', '▁', '₹', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x8A>']
Decoded: NAV ₹ 😊


1. ₹ is in the vocabulary

'₹' is a single token. TinyLlama's vocabulary includes the rupee symbol, so it gets its own token and ID. It did not need any special handling.


2. The emoji is not in the vocabulary, so it became bytes
😊  →  '<0xF0>', '<0x9F>', '<0x98>', '<0x8A>'

The emoji is stored in the computer as 4 bytes: F0, 9F, 98, 8A (these are hexadecimal numbers, which is what 0x means). The vocabulary has no token for this emoji, so the tokenizer uses one byte token per byte. The vocabulary includes 256 byte tokens, one for each possible byte, so any character can be represented this way.

This is called byte fallback: if a character is not in the vocabulary, fall back to its bytes.

3. That's why <unk> never appeared

Because of byte fallback, there is no character this tokenizer cannot represent. <unk> is still in the vocabulary (ID 0), but it is never needed for real text. And decode() joined the 4 bytes back into 😊, so the text came back exactly as it was.

The lone '▁' tokens
'▁', '₹'      → space, then ₹
'▁', '<0xF0>' → space, then the emoji's bytes

Normally the space is joined to the next word ('▁is'). But there is no vocabulary token for "space + ₹" or "space + emoji", so the space becomes its own token '▁', followed by the character.

**What to notice:** characters that are not in the vocabulary are not turned into `<unk>`. They are split into **byte tokens** like `<0xE2>`, and `decode()` joins them back into the original character. So this tokenizer can represent any text, and `<unk>` almost never appears. It still exists in the vocabulary (ID 0), which is why we could reuse it as padding in Step 5.

## Summary

| Token | TinyLlama | ID | Added automatically? | Job |
|---|---|---|---|---|
| BOS | `<s>` | 1 | Yes, at the start | Start of the text |
| EOS | `</s>` | 2 | No (added by the chat template or by us) | End of the text; stops generation |
| PAD | none (we used `<unk>`) | 0 | Only when `padding=True` | Filler for batches; ignored via attention mask |
| UNK | `<unk>` | 0 | Almost never used | Text the tokenizer cannot represent |

## Questions for the class

1. Why does the tokenizer add BOS automatically but not EOS?
2. What would happen during generation if a model never learned to produce EOS?
3. Why doesn't it matter which token we use as padding?
4. Why did `₹` and `😊` not become `<unk>`?

1. Why does the tokenizer add BOS automatically but not EOS?

BOS marks the start of the text, and every input has a start, so it is always correct to add it. EOS means "the text is finished". A prompt is not finished: we want the model to continue writing after it. If EOS were added to the end of the prompt, the model would read the text as already complete and would not answer properly. So EOS is added only where the text really ends: at the end of each answer in the training data, where the chat template adds it.

2. What would happen during generation if a model never learned to produce EOS?

It would never stop by itself. It would keep generating until it hit the max_new_tokens limit, and would usually repeat itself or drift into unrelated text after the useful answer. This is what the base model did in our demo: it used all 80 tokens and was cut off mid-sentence.

3. Why doesn't it matter which token we use as padding?

Because the model never reads the padding. The attention mask marks every padding position with 0, and those positions get zero weight, so they have no effect on the output. The padding token only fills space so that all rows in a batch have the same length. The one caution: avoid using the EOS token as padding when fine-tuning, because the code that removes padding from training could also remove the real EOS at the end of each answer, and the model would not learn to stop.

4. Why did ₹ and 😊 not become <unk>?

For different reasons:

₹ is in TinyLlama's vocabulary, so it became a normal token.
😊 is not in the vocabulary, but the tokenizer uses byte fallback: it split the emoji into its 4 bytes (<0xF0> <0x9F> <0x98> <0x8A>), and there is a byte token for every possible byte. Any character can be represented this way, so <unk> is never needed. decode() joins the bytes back into the emoji.